In [1]:
import os
import sys

PROJECT_MARKERS = ("src", "data", "prompts", "results")

def find_project_root(start_path):
    current = os.path.abspath(start_path)

    while True:
        if all(os.path.isdir(os.path.join(current, m)) for m in PROJECT_MARKERS):
            return current

        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError("Project root not found")

        current = parent


# ---- execution directory (cwd) ----
cwd = os.getcwd()

# ---- safe starting point ----
try:
    start_path = os.path.dirname(os.path.abspath(__file__))
except NameError:
    start_path = cwd


# ---- resolve canonical paths ----
project_root = find_project_root(start_path)

# ✅ THIS IS THE IMPORTANT PART
if project_root not in sys.path:
    sys.path.insert(0, project_root)

src_root     = os.path.join(project_root, "src", "daniel", "gemini")
data_root    = os.path.join(project_root, "data", "MAMS-ACSA", "raw", "data_jsonl", "annotated")
schemas_root = os.path.join(project_root, "data", "MAMS-ACSA", "raw", "data_jsonl", "schema")
prompts_root = os.path.join(project_root, "prompts", "daniel", "llama")
utils_root   = os.path.join(project_root, "utils")
results_root = os.path.join(project_root, "results", "daniel")

print(
    f"📂 cwd          : {cwd}\n"
    f"📂 Project root : {project_root}\n"
    f"📂 Source root  : {src_root}\n"
    f"📂 Data root    : {data_root}\n"
    f"📂 Prompts root : {prompts_root}\n"
    f"📂 Utils root   : {utils_root}\n"
    f"📂 Results root : {results_root}"
)

📂 cwd          : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/src/daniel/metrics
📂 Project root : /Users/hd/Desktop/RCS-Emotion-Prediction-2025
📂 Source root  : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/src/daniel/gemini
📂 Data root    : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/data/MAMS-ACSA/raw/data_jsonl/annotated
📂 Prompts root : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/prompts/daniel/llama
📂 Utils root   : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/utils
📂 Results root : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/results/daniel


In [2]:
import os
import json

# -----------------------------
# Paths
# -----------------------------
GOLD_PATH  = os.path.join(data_root, "edited_300_sample_cleaned_14jan.jsonl")
TRAIN_PATH = os.path.join(data_root, "train.jsonl")

WORKING_COPY_PATH = os.path.join(results_root, "sample_working_copy.jsonl")
INDEX_MAP_PATH    = os.path.join(results_root, "sample_working_copy_index_map.jsonl")
MISSING_PATH      = os.path.join(results_root, "sample_missing_from_train.jsonl")

print(f"Using GOLD : {GOLD_PATH}")
print(f"Using TRAIN: {TRAIN_PATH}")
print(f"Writing working copy : {WORKING_COPY_PATH}")
print(f"Writing index map    : {INDEX_MAP_PATH}")
print(f"Writing missing rows : {MISSING_PATH}")

# -----------------------------
# Build TRAIN input set (exact match)
# -----------------------------
train_inputs = set()
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        inp = row.get("input")
        if inp is not None:
            train_inputs.add(inp)

print(f"\nUnique TRAIN inputs: {len(train_inputs)}")

# -----------------------------
# Filter GOLD -> working copy
# Also write mapping (gold_index -> new_index)
# -----------------------------
kept = 0
missing = 0

with open(GOLD_PATH, "r", encoding="utf-8") as fin, \
     open(WORKING_COPY_PATH, "w", encoding="utf-8") as fout_work, \
     open(INDEX_MAP_PATH, "w", encoding="utf-8") as fout_map, \
     open(MISSING_PATH, "w", encoding="utf-8") as fout_missing:

    for gold_index, line in enumerate(fin):
        line = line.strip()
        if not line:
            continue

        g = json.loads(line)
        inp = g.get("input")

        if inp in train_inputs:
            new_index = kept

            # ✅ write the row you will later modify freely
            fout_work.write(json.dumps(g, ensure_ascii=False) + "\n")

            # ✅ write index mapping (anchor for stable IDs)
            fout_map.write(json.dumps({
                "gold_index": gold_index,
                "new_index": new_index,
                "input": inp
            }, ensure_ascii=False) + "\n")

            kept += 1
        else:
            # store missing with original gold index
            fout_missing.write(json.dumps({
                "gold_index": gold_index,
                "input": inp,
                "row": g
            }, ensure_ascii=False) + "\n")
            missing += 1

print("\nDONE")
print("----")
print(f"Kept rows (working copy) : {kept}")
print(f"Missing from TRAIN       : {missing}")
print(f"Working copy path        : {WORKING_COPY_PATH}")
print(f"Index map path           : {INDEX_MAP_PATH}")
print(f"Missing rows path        : {MISSING_PATH}")

Using GOLD : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/data/MAMS-ACSA/raw/data_jsonl/annotated/edited_300_sample_cleaned_14jan.jsonl
Using TRAIN: /Users/hd/Desktop/RCS-Emotion-Prediction-2025/data/MAMS-ACSA/raw/data_jsonl/annotated/train.jsonl
Writing working copy : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/results/daniel/sample_working_copy.jsonl
Writing index map    : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/results/daniel/sample_working_copy_index_map.jsonl
Writing missing rows : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/results/daniel/sample_missing_from_train.jsonl

Unique TRAIN inputs: 3151

DONE
----
Kept rows (working copy) : 292
Missing from TRAIN       : 7
Working copy path        : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/results/daniel/sample_working_copy.jsonl
Index map path           : /Users/hd/Desktop/RCS-Emotion-Prediction-2025/results/daniel/sample_working_copy_index_map.jsonl
Missing rows path        : /Users/hd/Desktop/RCS-Emotion-Prediction-20